In [1]:
import pandas as pd, category_encoders as ce

In [2]:
X = pd.DataFrame({"city": ["Paris"]*4 + ["Tokyo"]*2 + ["Lima"]})
y = pd.Series([1,1,1,0, 0,0, 1])


## Family 1 — Identity encodings (no target)

In [7]:
X

,city
0,Paris
1,Paris
2,Paris
3,Paris
4,Tokyo
5,Tokyo
6,Lima


In [8]:
ce.OrdinalEncoder().fit_transform(X) 

,city
0,1
1,1
2,1
3,1
4,2
5,2
6,3


In [9]:
ce.OneHotEncoder(use_cat_names=True).fit_transform(X)

,city_Paris,city_Tokyo,city_Lima
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,0,1,0
5,0,1,0
6,0,0,1


In [10]:
ce.BinaryEncoder().fit_transform(X)

,city_0,city_1
0,0,1
1,0,1
2,0,1
3,0,1
4,1,0
5,1,0
6,1,1


In [11]:
ce.BaseNEncoder(base=4).fit_transform(X)

,city_0
0,1
1,1
2,1
3,1
4,2
5,2
6,3


In [13]:
# adjacent indices differ by one bit
ce.GrayEncoder().fit_transform(X)  

,city_0,city_1
0,1,1
1,1,1
2,1,1
3,1,1
4,1,0
5,1,0
6,0,1


In [14]:
ce.GrayEncoder().fit_transform(X)

,city_0,city_1
0,1,1
1,1,1
2,1,1
3,1,1
4,1,0
5,1,0
6,0,1


In [15]:
ce.CountEncoder().fit_transform(X) 

,city
0,4
1,4
2,4
3,4
4,2
5,2
6,1


In [16]:
# hash(city) mod 8 → bucket
ce.HashingEncoder(n_components=8).fit_transform(X)

,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7
0,0,0,1,0,0,0,0,0
1,0,0,1,0,0,0,0,0
2,0,0,1,0,0,0,0,0
3,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,1,0
5,0,0,0,0,0,0,1,0
6,0,0,0,1,0,0,0,0


## Family 2 — Contrast codings (linear-model statistics)

## Family 3 — Target-based encoders

In [22]:
ce.TargetEncoder(min_samples_leaf=2, smoothing=1).fit_transform(X, y)
# y:[1,1,1,0, 0,0, 1]

,city
0,0.728714
1,0.728714
2,0.728714
3,0.728714
4,0.285714
5,0.285714
6,0.686689


In [20]:
ce.MEstimateEncoder(m=2).fit_transform(X, y)
# y:[1,1,1,0, 0,0, 1]

,city
0,0.690476
1,0.690476
2,0.690476
3,0.690476
4,0.285714
5,0.285714
6,0.714286


In [23]:
ce.JamesSteinEncoder().fit_transform(X, y)
# y:[1,1,1,0, 0,0, 1]

,city
0,0.75
1,0.75
2,0.75
3,0.75
4,0.00
5,0.00
6,1.00


### WOE
$$\mathrm{WOE}_c = \ln \frac{P(c \mid y=1)}{P(c \mid y=0)} = \ln \frac{n_{1c}/n_1}{n_{0c}/n_0}$$
$$\operatorname{logit} P(y=1 \mid x) = \operatorname{logit}(\bar{y}) + \sum_{j} \mathrm{WOE}_j(x_j)$$
$$\mathrm{IV} = \sum_{c} \bigl(P(c \mid 1) - P(c \mid 0)\bigr)\,\mathrm{WOE}_c = D_{\mathrm{KL}}(P_1 \,\|\, P_0) + D_{\mathrm{KL}}(P_0 \,\|\, P_1)$$

In [24]:
# log P(c|y=1) / P(c|y=0)
# (log-odds ratio)
ce.WOEEncoder().fit_transform(X, y)
# y:[1,1,1,0, 0,0, 1] prior: 4/7 ≈ 0.571

,city
0,0.510826
1,0.510826
2,0.510826
3,0.510826
4,-1.280934
5,-1.280934
6,0.000000


### QuantileEncoder

$$\tilde{x}_c = \frac{q_p(y \mid c)\cdot n_c + q_p(y)\cdot m}{n_c + m}$$
where $q_p$ is the $p$-th quantile (default: median)

In [25]:
Xr = pd.DataFrame({"plan": ["A"]*5 + ["B"]*4 + ["C"]})
yr = pd.Series([10,12,11,13,300,  20,22,25,18,  50])

In [26]:
ce.QuantileEncoder(quantile=0.5, m=1.0).fit_transform(Xr, yr)

,plan
0,13.166667
1,13.166667
2,13.166667
3,13.166667
4,13.166667
5,20.600000
6,20.600000
7,20.600000
8,20.600000
9,34.500000


In [27]:
ce.SummaryEncoder(quantiles=[0.25, 0.5, 0.75], m=1.0).fit_transform(Xr, yr)

,plan_25,plan_50,plan_75
0,11.208333,13.166667,14.875
1,11.208333,13.166667,14.875
2,11.208333,13.166667,14.875
3,11.208333,13.166667,14.875
4,11.208333,13.166667,14.875
5,18.050000,20.600000,23.050
6,18.050000,20.600000,23.050
7,18.050000,20.600000,23.050
8,18.050000,20.600000,23.050
9,31.125000,34.500000,37.125
